In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split 
from sklearn.metrics import precision_recall_curve, f1_score
from toad.metrics import KS, AUC 
from toad.plot import bin_plot, badrate_plot
import xgboost as xgb 
import pandas as pd
import seaborn as sns 
import numpy as np
import matplotlib.pyplot as plt
import toad
import pickle

In [ ]:
df = pd.read_excel('UPI_trn V2.xlsx')

In [ ]:
df.head(5)
df.shape

In [ ]:
df.head()

In [ ]:
import pandas as pd

# Assuming df is your DataFrame
unique_values = df['PROVINCE'].unique()
print(unique_values) 

In [ ]:
unique_count = df['OCCUP_DET_DESC'].nunique()
print(f'The number of unique types in OCCUP_DET_DESC is: {unique_count}')

In [ ]:
df = df.drop(columns=['CUSTOMER_ID', 'PROVINCE', 'DISTRICT', 'OCCUP_TYPE_DESC','TRN_CNT','TRN_AMOUNT_USD','AVG_BALANCE_GROUP','IS_USAGE'])


In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
numeric_cols = [
    'AGE',
    'MB_INT_TRF_CNT',
    'MB_LOCAL_TRF_CNT',
    'MB_LOCAL_TRF_AMOUNT_USD',
    'MB_PAYMNENT_CNT',
    'MB_PAYMNENT_CNT.1',
    'AVG_BALANCE_USD'
]

categorical_cols = [
    'GENDER',
    'MARITAL_STATUS',
    'PROVINCE',
    'OCCUP_DET_DESC',
    
]

target_col = 'NOT_USE'


In [ ]:
df[numeric_cols] = df[numeric_cols].fillna(0)


In [ ]:
zero_cols = [
    'MB_INT_TRF_CNT', 'MB_INT_TRF_AMOUNT_USD', 'MB_LOCAL_TRF_CNT',
    'MB_LOCAL_TRF_AMOUNT_USD', 'MB_PAYMNENT_CNT', 'MB_PAYMNENT_CNT.1'
]

df = df[~(df[zero_cols].sum(axis=1) == 0)]



In [ ]:
from sklearn.preprocessing import QuantileTransformer

df['AVG_BALANCE_USD'] = df['AVG_BALANCE_USD'].clip(lower=0)

lower_cap = df['AVG_BALANCE_USD'].quantile(0.01)
upper_cap = df['AVG_BALANCE_USD'].quantile(0.99)
df['AVG_BALANCE_USD_CAPPED'] = df['AVG_BALANCE_USD'].clip(lower=lower_cap, upper=upper_cap)


df['AVG_BALANCE_USD_LOG'] = np.log1p(df['AVG_BALANCE_USD'])


In [ ]:
df.columns

In [ ]:
df = df.drop(columns=['AVG_BALANCE_USD','AVG_BALANCE_USD_CAPPED'])


In [ ]:
df['AVG_BALANCE_USD_LOG'].describe()

In [ ]:
numeric_cols = [
    'AGE','MB_INT_TRF_CNT', 'MB_INT_TRF_AMOUNT_USD',
    'MB_LOCAL_TRF_CNT', 'MB_LOCAL_TRF_AMOUNT_USD',
    'MB_PAYMNENT_CNT', 'MB_PAYMNENT_CNT.1',
    'AVG_BALANCE_USD_LOG'
]


In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.head()

num_cols = [
    'AGE', 'MB_INT_TRF_CNT', 'MB_INT_TRF_AMOUNT_USD',
    'MB_LOCAL_TRF_CNT', 'MB_LOCAL_TRF_AMOUNT_USD',
    'MB_PAYMNENT_CNT', 'MB_PAYMNENT_CNT.1', 'AVG_BALANCE_USD'
]


categorical_cols = [
    'GENDER',
    'MARITAL_STATUS',
    'PROVINCE',
    'OCCUP_DET_DESC',

]

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

correlation_matrix = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
plt.title('Correlation Heatmap', fontsize=16)
plt.show()

In [ ]:
 df = df.drop(columns=['MB_INT_TRF_AMOUNT_USD'])


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
df['AGE'].hist(bins=30)
plt.title('Histogram')

plt.subplot(1,2,2)
df.boxplot(column='AGE')
plt.title('Boxplot')

plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='MB_INT_TRF_CNT', hue='NOT_USE', bins=30, kde=True, palette='Set2')
plt.title("Distribution of MB_INT_TRF_CNT by IS_USAGE (0 = No, 1 = Yes)", fontsize=13, fontweight='bold')
plt.xlabel("MB_INT_TRF_CNT")
plt.ylabel("Number of Customers")
plt.legend(title="NOT_USE", labels=["0 = No Usage", "1 = Usage"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
top_outliers = list(df.MB_INT_TRF_CNT.sort_values()[-6:].index)
df = df.drop(top_outliers)

In [ ]:
top_outliers = list(df.MB_LOCAL_TRF_CNT.sort_values()[-6:].index)
df = df.drop(top_outliers)

In [ ]:
print(df['MB_LOCAL_TRF_CNT'].describe())
print(df['MB_LOCAL_TRF_CNT'].value_counts().head(10))
print(df.groupby('NOT_USE')['MB_LOCAL_TRF_CNT'].describe())

In [ ]:
  #'MB_LOCAL_TRF_CNT': [5,10,30,80,200,350]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
df['MB_LOCAL_TRF_AMOUNT_USD'].hist(bins=30)
plt.title('Histogram')

plt.subplot(1,2,2)
df.boxplot(column='MB_LOCAL_TRF_AMOUNT_USD')
plt.title('Boxplot')

plt.show()


In [ ]:
top_outliers = list(df.MB_LOCAL_TRF_AMOUNT_USD.sort_values()[-2:].index)
df = df.drop(top_outliers)

In [ ]:
print(df['MB_LOCAL_TRF_AMOUNT_USD'].describe())
print(df['MB_LOCAL_TRF_AMOUNT_USD'].value_counts().head(10))
print(df.groupby('NOT_USE')['MB_LOCAL_TRF_AMOUNT_USD'].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
df['MB_PAYMNENT_CNT'].hist(bins=30)
plt.title('Histogram')

plt.subplot(1,2,2)
df.boxplot(column='MB_PAYMNENT_CNT')
plt.title('Boxplot')

plt.show()


In [ ]:
top_outliers = list(df.MB_PAYMNENT_CNT.sort_values()[-8:].index)
df = df.drop(top_outliers)

In [ ]:
print(df['MB_PAYMNENT_CNT'].describe())
print(df['MB_PAYMNENT_CNT'].value_counts().head(20))
print(df.groupby('NOT_USE')['MB_PAYMNENT_CNT'].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
df['MB_PAYMNENT_CNT.1'].hist(bins=30)
plt.title('Histogram')

plt.subplot(1,2,2)
df.boxplot(column='MB_PAYMNENT_CNT.1')
plt.title('Boxplot')

plt.show()


In [ ]:
#top_outliers = list(df.MB_PAYMNENT_CNT.1.sort_values()[-8:].index)
#df = df.drop(top_outliers)

In [ ]:
print(df['MB_PAYMNENT_CNT.1'].describe())
print(df['MB_PAYMNENT_CNT.1'].value_counts().head(20))
print(df.groupby('NOT_USE')['MB_PAYMNENT_CNT.1'].describe())

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12,4))

# plt.subplot(1,2,1)
# df['AVG_BALANCE_USD'].hist(bins=30)
# plt.title('Histogram')

# plt.subplot(1,2,2)
# df.boxplot(column='AVG_BALANCE_USD')
# plt.title('Boxplot')

# plt.show()


top_outliers = list(df.AVG_BALANCE_USD.sort_values()[-20:].index)
df = df.drop(top_outliers)

print(df['AVG_BALANCE_USD'].describe())
print(df['AVG_BALANCE_USD'].value_counts().head(20))
print(df.groupby('IS_USAGE')['AVG_BALANCE_USD'].describe())

In [ ]:
df.columns

In [ ]:
feature_bins = {
    'AGE': [25,30,35,40],
    'MB_INT_TRF_CNT': [0, 1, 5, 20, 50],
    'AVG_BALANCE_USD_LOG': [1,2,3],
    'MB_PAYMNENT_CNT.1': [1,20,90],
    'MB_PAYMNENT_CNT': [1,10,25],
    'MB_LOCAL_TRF_AMOUNT_USD': [50,300,1000],
    'MB_LOCAL_TRF_CNT': [10,30,100],
 #   'AVG_BALANCE_USD_CAPPED': [0, 100, 1000, 5000]
}
#'MB_LOCAL_TRF_AMOUNT_USD': [10,100,700,3000,6000,12000],

#'MB_LOCAL_TRF_AMOUNT_USD': [100,1000,2000],

In [ ]:
def label_info(label_column):
    default_true = str(round(sum(label_column == True) / len(label_column), 2)*100)
    default_false = str(round(sum(label_column == False) / len(label_column), 3)*100)

    print('There are {} records in total.'.format(len(label_column)))
    print('Default:')
    print('Counts {}'.format(sum(label_column == True)))
    print('Proportion {}'.format(default_true + '%'))
    print('----------------------------------------')
    print('Not Default:')
    print('Counts {}'.format(sum(label_column == False)))
    print('Proportion {}'.format(default_false + '%'))
    
    plt.figure(figsize=(5,5))
    sns.barplot(x = label_column.value_counts().index, y = label_column.value_counts())
    plt.title('Non-Default vs Default')
    plt.ylabel('Counts')
    return default_true

label_info(df['NOT_USE'])

In [ ]:
exclude_list = ['NOT_USE']

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size = 0.2, random_state = 42, stratify=df['NOT_USE'])

print('Train Default Counts: {}'.format(train.NOT_USE.value_counts()))
print('-------------------------------------------------')
print('Test Default Counts: {}'.format(test.NOT_USE.value_counts()))

In [ ]:
test

In [ ]:
toad.detect(train)

In [ ]:
toad.quality(train,'NOT_USE')

In [ ]:
train_selected, drop_list = toad.selection.select(
    frame = train,
    target = train['NOT_USE'],
    empty = 0.7, #Drop columns with >70% missing values
    iv = 0.02, # Drop columns with IV < 0.02 (weak predictors)
    corr = 1, # Drop perfectly correlated (redundant) columns
    return_drop = True,
    exclude = exclude_list)

print("Keep:", train_selected.shape[1],
     "Drop Empty:", len(drop_list['empty']),
     "Drop IV:", len(drop_list['iv']),
     "Drop CORR:", len(drop_list['corr']))

In [ ]:
print(drop_list)
train_selected.head()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(train_selected.select_dtypes(include=['number']).corr(), 
            annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.show()


In [ ]:
 #train_selected = train_selected.drop(columns=['MB_LOCAL_TRF_CNT'])

In [ ]:
train_selected.columns

In [ ]:
train_selected.NOT_USE.value_counts()

In [ ]:
import os
os.environ["SCIPY_ARRAY_API"] = "1"

In [ ]:
df

In [ ]:
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import BorderlineSMOTE
import pandas as pd

X = train_selected.drop(columns=['NOT_USE'])
y = train_selected['NOT_USE']


X_encoded = X.copy()
for col in X_encoded.select_dtypes(include=['object']).columns:
    X_encoded[col] = LabelEncoder().fit_transform(X_encoded[col])


smote = BorderlineSMOTE(sampling_strategy=1.0, random_state=42)
x_bal, y_bal = smote.fit_resample(X_encoded, y)


print("Before:", y.value_counts().to_dict())
print("After:", pd.Series(y_bal).value_counts().to_dict())


In [ ]:
import pandas as pd


print(pd.Series(y_bal).value_counts())


In [ ]:
import toad
from toad.transform import Combiner
import pandas as pd

# X = your features, y = your target
# Use x_bal and y_bal as you did before
# Example:
# x_bal = df.drop(columns=['target'])
# y_bal = df['target']

combiner = Combiner()
combiner.fit(x_bal, y_bal, method='quantile', n_bins=6)
update_bins = combiner.export()

cat_high_card_cols = ['OCCUP_DET_DESC'] #['OCCUP_DET_DESC', 'PROVINCE']

cat_combiner = Combiner()
cat_combiner.fit(
    X=x_bal[cat_high_card_cols],
    y=y_bal,
    method='chi',
    min_samples=0.05
)
cat_bins = cat_combiner.export()
print("Chi-merge binning result:", cat_bins)

update_bins.update(cat_bins)
combiner.set_rules(update_bins)

x_binned = combiner.transform(x_bal)



In [ ]:
update_bins = combiner.export()

for feature, bins in feature_bins.items():
    update_bins[feature] = bins

combiner.set_rules(update_bins)
update_bins

In [ ]:
train_selected_bin = combiner.transform(x_bal)
test_bin = combiner.transform(test[train_selected_bin.columns])

In [ ]:
train_selected_bin.columns

In [ ]:
train_selected_bin_full = train_selected_bin.copy()
train_selected_bin_full['NOT_USE'] = y_bal

cols = [col for col in train_selected_bin_full.columns if col != 'NOT_USE']

for i in cols:
    bin_plot(train_selected_bin_full, x=i, target='NOT_USE')
 #'AGE': [15,25,30,35,40],


In [ ]:
print(train_selected_bin.columns)

In [ ]:
train_binned = combiner.transform(train_selected_bin)


In [ ]:
t = toad.transform.WOETransformer()
train_woe = t.fit_transform(
    X = train_selected_bin_full,
    y = train_selected_bin_full['NOT_USE'],
    exclude = exclude_list
)
test_woe = t.transform(test_bin)
t


In [ ]:
test_woe['NOT_USE'] = test['NOT_USE'].values
final_data_woe = pd.concat([train_woe, test_woe])


In [ ]:
final_data_woe

In [ ]:
# Save 2: Woe Transform for Scorecard
filename = 'WOE.pkl'
pickle.dump(t, open(filename, 'wb'))

In [ ]:
features_use = [feat for feat in final_data_woe.columns if feat not in exclude_list]

def output_iv(train_selected_bin_full, label_col):
    important_features = toad.quality(train_selected_bin_full, label_col, iv_only = True) # Only return the IV values 
    important_features = important_features['iv']
    important_features = important_features.reset_index()
    important_features.columns = ['Column Name', 'iv']
    return important_features 

loan_data_iv = output_iv(final_data_woe[features_use+['NOT_USE']], 'NOT_USE')
len(features_use)
#train_selected

In [ ]:
loan_data_iv

In [ ]:
# Save 4: Information Value After WOE Transformation
loan_data_iv.to_csv('IV2.csv', index = False)

In [ ]:
features_use = [f for f in features_use if f not in {'MB_INT_TRF_CNT','MB_INT_TRF_AMOUNT_USD'}]
features_use

In [ ]:
x_train = train_woe[features_use]
y_train = train_woe['NOT_USE']
x_test = test_woe[features_use]
y_test = test_woe['NOT_USE']

threshold = 0.65 

In [ ]:
def check_train_test_auc(x_train, y_train, x_test, y_test):
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',  
        eval_metric='auc',  
        scale_pos_weight=10,  
        learning_rate=0.04,  
        n_estimators=450,  
        max_depth=6,  
        subsample=0.8,  
        colsample_bytree=0.8,  
        random_state=42,
        reg_lambda=40,  
        reg_alpha=15,  
        gamma=0.5
    )

    xgb_model.fit(x_train, y_train)

    # Predict probabilities
    pred_train_proba = xgb_model.predict_proba(x_train)[:, 1]
    pred_test_proba = xgb_model.predict_proba(x_test)[:, 1]

    x_train['Predicted_Score'] = pred_train_proba
    x_test['Predicted_Score'] = pred_test_proba
    
    # Optimize the threshold for the best F1 score on the test set
    precisions, recalls, thresholds = precision_recall_curve(y_test, pred_test_proba)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-9)
    best_index = np.argmax(f1_scores)
    best_threshold = thresholds[best_index]
    print(f"Optimal Threshold: {best_threshold:.4f}")
    print('____________________________________________')
    
    # Calculate KS and AUC
    ks_train = KS(pred_train_proba, y_train)
    auc_train = AUC(pred_train_proba, y_train)
    ks_test = KS(pred_test_proba, y_test)
    auc_test = AUC(pred_test_proba, y_test)
    print(f"Train KS: {ks_train:.4f}")
    print(f"Train AUC: {auc_train:.4f}")
    print(f"Test KS: {ks_test:.4f}")
    print(f"Test AUC: {auc_test:.4f}")
    print('____________________________________________')

    # Get feature importances based on gain, weight, and cover
    booster = xgb_model.get_booster()
    gain = booster.get_score(importance_type='gain')
    weight = booster.get_score(importance_type='weight')
    cover = booster.get_score(importance_type='cover')

    gain_df = pd.DataFrame(list(gain.items()), columns=['Feature_Name', 'Gain_Importance'])
    weight_df = pd.DataFrame(list(weight.items()), columns=['Feature_Name', 'Weight_Importance'])
    cover_df = pd.DataFrame(list(cover.items()), columns=['Feature_Name', 'Cover_Importance'])

    feat_important = pd.merge(gain_df, weight_df, on='Feature_Name', how='outer').merge(cover_df, on='Feature_Name', how='outer')
    feat_important.fillna(0, inplace=True)

    for imp_type in ['Gain_Importance', 'Weight_Importance', 'Cover_Importance']:
        sorted_features = feat_important.sort_values(imp_type, ascending=False)
        plt.figure(figsize=(12, 8))
        sns.barplot(x=sorted_features[imp_type], y=sorted_features['Feature_Name'])
        plt.title(f'Feature Importance based on {imp_type}', size=18)
        plt.xlabel(imp_type.replace('_', ' '), size=15)
        plt.ylabel('Feature Name', size=15)
        plt.xticks(rotation=45)
        plt.grid()
        plt.show()

    return xgb_model, pred_train_proba

In [ ]:
xgb_model = check_train_test_auc(x_train, y_train, x_test, y_test)

In [ ]:
import pickle

# Save 3: XGB model
filename = 'XGB_model.pkl'
pickle.dump(t, open(filename, 'wb'))

In [ ]:
test_binned = combiner.transform(test_bin)
test_woe = t.transform(test_binned)


In [ ]:
card = toad.ScoreCard(
    combiner=combiner,
    transer=t,
    class_weight='balanced',
    C=0.1,
    base_score=800,
    base_odds=20,
    pdo=20,
    rate=2
)

card.fit(train_woe[features_use], train_woe['NOT_USE'])

In [ ]:
with open("scorecard.pkl", "wb") as f:
    pickle.dump(card, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Apply on test data
test['Score'] = card.predict(test)
test['Score'].describe()

In [ ]:
final_card_score = card.export()
len(final_card_score)

In [ ]:
# Transform scorecard into DF
keys = list(card.export().keys())
score_card_df = pd.DataFrame()

score_card_list = [] 

for n in keys:
    temp = pd.DataFrame.from_dict(final_card_score[n], orient='index')
    temp = temp.reset_index()
    temp.columns = ['Binning', 'Score']
    temp['Variable'] = n
    temp = temp[['Variable', 'Binning', 'Score']]
    
    score_card_list.append(temp) 

score_card_df = pd.concat(score_card_list, ignore_index=True)

score_card_df.to_csv('feature_binning_score2.csv', index = False)
score_card_df.head(65)

In [ ]:
test

In [ ]:
import matplotlib.pyplot as plt
import math

w = 20
n = math.ceil((test['Score'].max() - test['Score'].min()) / w)

plt.figure(figsize=(12, 10))
plt.hist(test[test.NOT_USE == 1]['Score'], alpha=0.5, label='Usage = 1', bins=n)
plt.hist(test[test.NOT_USE == 0]['Score'], alpha=0.5, label='Usage = 0', bins=n)
plt.legend(loc='upper left')
plt.title('Loan Score Distribution: Test Data', size=15)
plt.xlabel('Score')
plt.ylabel('Count')
plt.show()


In [ ]:
test['Score'].describe()

In [ ]:
def reverse_score(old_sc, var_sc):
    return var_sc-old_sc

In [ ]:
def get_loan_level(test, target_score='Score', out_col='Level'):
    bins = [550, 650, 750, 850, 1000]  
    labels = [
        'Poor',         
        'Fair',       
        'Good',          
        'Excellent',       
    ]
    min_sc = test[target_score].min()
    max_sc = test[target_score].max()
    test['new_score'] = test.apply(lambda x:reverse_score(x['Score'],max_sc+min_sc),axis=1)
    test[out_col] = pd.cut(test['new_score'], bins=bins, labels=labels, right=True)  # Categorize scores
    
    return test
    
test = get_loan_level(test)

In [ ]:
test.to_csv('Prediction2.csv', index = False)

In [ ]:
var_sc = test['Score'].min()+test['Score'].max()

In [ ]:
test['new_score'] = test.apply(lambda x:reverse_score(x['Score'],var_sc),axis=1)

In [ ]:
test.to_csv('Prediction2.csv', index = False)